# Week 7 - NYT API [something something]
**Name:Akua Amponsah**
**Date: October 12, 2026**

In [4]:
#first we import everything we need for the task at hand

import requests # this gives us the ability to request data from remote websites
from time import sleep #this gives us the ability to implement time delays in our code
from getpass import getpass #this is a password-protection feature for python notebooks

api_key = getpass ("Paste your API key here!")

Paste your API key here! ········


In [15]:
query = "A.I." #This is the search term we are looking for in any of the metadata
begin_date = "19000101" #this is the begin date for our search (was initially 20260901 September 1 2026 but wanna go back real real far)
end_date = "20260930" #this is the end date for our search (September 30 2026)

sort = "oldest"

url = f"https://api.nytimes.com/svc/search/v2/articlesearch.json?q={query}&begin_date={begin_date}&end_date={end_date}&api-key={api_key}&sort={sort}"
response = requests.get(url)

print(response.status_code)

200


Pulling the actual documents out of the API response:
 - First we need to parse the response as JSON, then we need to find the key that contains the article info

In [16]:
nyt_response_parsed = response.json() #interpret response as Python-readable JSON
print(nyt_response_parsed.keys()) #tells me what keys are in the raw response
print(nyt_response_parsed["response"].keys()) #tells me what keys are in the "response" part
nytimes_docs = nyt_response_parsed["response"]["docs"] #tells me what keys are in the "docs" key, which is a child of "response"
print(len(nytimes_docs))

dict_keys(['status', 'copyright', 'response'])
dict_keys(['docs', 'metadata'])
10


In [17]:
print(nytimes_docs[0]["headline"])
print(nytimes_docs[0]["headline"]["main"])
print(nytimes_docs[0]["keywords"])
print()
print(nytimes_docs[0]["abstract"])

{'main': 'LEGISLATIVE NOTES.', 'kicker': '', 'print_headline': 'LEGISLATIVE NOTES.'}
LEGISLATIVE NOTES.
[{'name': '', 'value': '', 'rank': 1}, {'name': '', 'value': '', 'rank': 2}]

Circulars Captured and Five Prisoners Taken in New York, (Brooks, Jackson, Gilbert, Lie, Walker)


## Web scraping with BeautifulSoup!
Below are the libraries I need to import before I begin web scraping:

In [19]:
import csv
import json
import os
import requests
from time import sleep
from bs4 import BeautifulSoup #the scraping library

Below is some example HTML that we can traverse using the BeautifulSoup functions!

In [20]:
html = """
<html>
  <body>
    <h1>Adeline Ravoux</h1>
    <p class="artist">Vincent van Gogh</p>
    <p class="date">1890</p>
    <a href="/art/1958.31">Artwork page</a>
  </body>
</html>
"""

soup = BeautifulSoup(html, "html.parser")  # html.parser is the built-in parser that comes with python
print(type(soup))

<class 'bs4.BeautifulSoup'>


In [21]:
title = soup.find("h1") #finds the first element that uses the h1 tag
print(title)

<h1>Adeline Ravoux</h1>


In [22]:
print(title.get_text())

Adeline Ravoux


In [23]:
paragraph_elements = soup.find_all("p")
print(paragraph_elements)

[<p class="artist">Vincent van Gogh</p>, <p class="date">1890</p>]


In [26]:
artist = soup.find("p", class_="artist") #underscore under class because it's a reserved word in Python
print(artist.get_text())
date = soup.find("p", class_="date")
print(date.get_text())

Vincent van Gogh
1890


## Understanding "robots" permissions
We can view the permissions that a website grants to do things like automatically access data with a "bot" by looking at its `robots.txt` file. It always lives at the robots.txt file in the root directory of the domain (for instance, www.clevelandart.org/robots.txt)

In [ ]:
headers = {"User-Agent": "INFO664-class-project (educational)"}   # you can call this whatever you want, it shows up on the scraped website's logs as who you are (ie not a Google/AI bot or web browser)

os.makedirs("scraped_pages", exist_ok=True)  # make a new folder to hold our scraped website HTML data. exist_ok = True means, if the directory already exists, do nothing and don't return an error

def fetch_page(url, filename):  # declaring a new function, which takes a URL to be scraped, and a destination filename, as arguments
    if os.path.exists(filename):  # check to see if file already exists, if so, don't download it from the web, load it instead
        with open(filename, "r", encoding="utf-8") as file:
            return file.read()  # in this case, return provides the read file data, and escapes from the function, ignoring everything below
#return function does two things: (1) sends that data back into the function and (2) escaping the function so it no longer runs
    response = requests.get(url, headers=headers) # grab the file, using our pre-defined headers as the header data

    if not response.ok: # if the response reports an error
        print(url, "failed with status code", response.status_code) # tell us why it failed
        return None # return a "None" object, since there's nothing usable in the response, and escape from this function
    with open(filename, "w",  encoding="utf-8") as file:
        file.write(response.text)

    sleep(10) # wait 10 seconds, since our robots.txt file asks for a 10 second Crawl-delay
    return response.text

